# DPO results — samples

SFT (`sft_instruct_2026-09-22_17-53-47`, the reference) against DPO
(`dpo_instruct_2026-09-22_19-24-50`: 1999 pairs, beta 0.1, lr 1e-5, 500 steps).
Same held-out prompt and same seed for both. Each story is followed by its reward
breakdown and the target words it missed.

Scoreboard on 300 held-out prompts: reward **0.653 -> 0.759** sampled, **0.681 -> 0.781**
greedy, stories **+22% longer**. Numbers are in the README; this notebook is for reading
what changed.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src" / "video"))

import torch

from checkpoint import load_checkpoint
from generate import generate
from instruct import Instruct, _words, checks, stem, target_words
from tokenizer import ENDOFTEXT

CKPT = root / "artifacts" / "checkpoints"
models = {
    "SFT": load_checkpoint(CKPT / "sft_instruct_2026-09-22_17-53-47.pt", "cuda")[0],
    "DPO": load_checkpoint(CKPT / "dpo_instruct_2026-09-22_19-24-50.pt", "cuda")[0],
}
task = Instruct()
tok, pool = task.tok, task.prompts()


def story(model, prompt, seed, temperature):
    x = torch.tensor([tok.encode(prompt)], device="cuda")
    room = model.block_size + 1 - x.size(1)
    g = torch.Generator(device="cuda").manual_seed(seed)
    out = generate(
        model, x, min(384, room), temperature=temperature,
        use_cache=True, generator=g, stop=task.eot,
    )  # fmt: skip
    return tok.decode(out[0, x.size(1) :].tolist())


def show(prompt, seed=0, temperature=1.0):
    print(prompt)
    want = target_words(prompt)
    for name, m in models.items():
        text = story(m, prompt, seed, temperature)
        body = text.split(ENDOFTEXT)[0].strip()
        c = checks(prompt, text, task.idf)
        missed = sorted(want - {stem(w) for w in _words(body)})
        scores = "  ".join(f"{k} {v:.2f}" for k, v in c.items())
        print(f"--- {name} | reward {task.reward(prompt, text):.2f} | {scores} | "
              f"{len(body.split())} words")
        print(f"    missed: {', '.join(missed) or '-'}")
        print(body, "\n")

## 1. The waffle prompt

The prompt the rarity weighting was built on: SFT stories kept dropping the plot words
*bitten / hospital / stitches*. Sampled at t=1.0.

Result: DPO **0.81 vs 0.71** and now says *dog, hospital, bitten*, but as keywords, not plot: "The bite was very bitten, and soon Lily was not bitten anymore." The reward counts the words; nothing checks they make sense.

In [2]:
waffle = next(p for p in pool if "waffle" in p and "stitches" in p)
show(waffle, seed=0)

Words: meet, waffle, new
Summary: Lily wants a waffle but gets bitten by a dog while playing with a toy car and has to go to the hospital for stitches.
Story:



--- SFT | reward 0.71 | stop 1.00  words 0.57 | 144 words
    missed: bitte, dog, hospi, meet
Once upon a time, there was a little girl named Lily. She was very excited because her mommy said they were going to the waffle shop. Lily loved waffles, especially the one with big, sharp stitches.
But when they got to the store, Lily saw a new toy bird and she wanted to play with it. But her mommy told her to be careful and not touch the car. Lily forgot and played with the toy car, but she noticed something strange. The car was slipping and not exciting. It was making a funny sound.
Lily ran over to the car and played with it like she wanted to have a regular toy. She forgot all about the new toy and had a new toy to play with. From then on, she went back to the car shop to play with her new toy. 



--- DPO | reward 0.81 | stop 1.00  words 0.72 | 193 words
    missed: meet, stitc
Once upon a time, there was a little girl named Lily. She was very excited to have a new waffle for her mom's birthday. The waffle was amazing. Every time her mom got a waffle, she put a big smiley face on the waffle and made a yummy sound.
Lily went to the fridge to find a snack. As she was looking, she saw a big dog. The dog was playing with a toy car. Lily wanted the animal toy, but her mom said, "No, Lily! That's not your waffle. You can get a new one." Lily was sad.
Then, something unexpected happened. Lily's mom went to the hospital and had to get a new waffle for her birthday. The bite was very bitten, and soon Lily was not bitten anymore. But when she tried the special waffle, she successfully became a cute little dog once again!
Everyone in the hospital cheered for Lily because she was a little deceiving the dog. The dog wanted to help Lily learn and grow big and strong. Lily and her mom shared t

## 2. Four held-out prompts, sampled (t=1.0)

The setting the pairs were drawn in.

Result: DPO wins 3 of 4, ties the last. The stadium prompt is the real gain: SFT never stopped (reward 0) and wandered off; DPO goes to the stadium and watches the race. Elsewhere the extra words are often inserted, not used ("She stole my favorite doll... I don't steal her"). DPO stories are longer in every pair.

In [3]:
picks = [3, 57, 121, 240]
for i in picks:
    show(pool[i], seed=i)
    print("=" * 100)

Words: point, stadium, fast
Summary: Tom, Lily and their dad go to the stadium to watch cars race. They cheer for their favorite cars and have a fun time.
Features: Dialogue
Story:



--- SFT | reward 0.00 | stop 0.00  words 0.70  dialogue 1.00 | 282 words
    missed: cheer, favor, race, time
Tom and Lily were twins who liked to play with their toy cars. They had many vehicles in their room, like blue ones, red ones, yellow ones, and green ones. Tom liked a red car that made loud noises and had red stripes. Lily liked a blue car that had fast stripes.
One day, they asked their dad if they could go to the stadium to watch cars. Dad smiled and said yes. He helped them put on their shoes and coats and took them to the car. On the way, Tom pointed to a big sign that said "Welcome to the Catch". He said, "Welcome to the Catch!"
They got in the car and drove to the stadium. It was very noisy and crowded. Tom and Lily held hands and looked out the window. They saw many other cars and people on the roads and houses. They saw a fence that said "Welcome to the askedion The Park". They saw cows and horses on the farm. They saw vests and coats and hats on the horses.
"Are we th

--- DPO | reward 0.89 | stop 1.00  words 0.77  dialogue 1.00 | 217 words
    missed: cheer, dad, watch
Once upon a time, there was a little boy named Tom and Lily. They lived near a big stadium where two cars raced each other. One car was very fast and one car was also in the stadium. They wanted to see which car to try.
One day, Tom said to Lily, "I want to see the fast car race. Do you want to see my favorite car?" Lily said, "Yes, I want to see the fast car race. Let's go!"
They started to point to cars as fast as they could. The fast cars passed by quickly. Tom pointed to a red car and said, "Lily, you are already on the fast car. Can you help me move?" Lily said, "Yes, I will help you!"
They pushed the red car as fast as they could. Tom said, "I like this car, Lily! Let's go see it both!" So they went with the stadium to see the fast cars.
They took their turns to control the toy cars. Soon, Tom's favorite car was the slowest. Lily's toy car went as fast as it could race. They lau

--- SFT | reward 0.79 | stop 1.00  words 0.58  dialogue 1.00 | 161 words
    missed: diffe, open, park, steal, stole, taugh
Once upon a time, there was a little girl named Lily. She was playing with her dolls when suddenly her favorite doll got caught in a tree. She tried to get it down, but the tree was too tall and she couldn't reach it.
Lily went to her friend Timmy and said, "Help me get my doll down." Timmy said, "Sure, maybe we can both save it!" They both tried very hard to reach the doll, but they couldn't get it down.
Lily's mom came and said, "Don't worry, Lily, we can't take things that don't belong to us. We have to ask the police if we can have it." Lily and Timmy were happy that they didn't have to give up the doll. They learned that it's important to ask before taking something that doesn't belong to you.
The moral of the story is to always be a good friend and ask before taking something that doesn't belong to you. 



--- DPO | reward 0.86 | stop 1.00  words 0.73  dialogue 1.00 | 259 words
    missed: doesn, impor, open, taugh
Once upon a time, there was a little girl named Lily. She was playing with her dolls at her house. Suddenly, Lily's friend Timmy came to the park. "Hey Lily, I have a different doll. She stole my favorite doll when I was playing with her," said Timmy.
Lily wanted to see the doll up close, so she asked Timmy. But Timmy said, "No, I don't steal her. I want her back." Timmy took the doll and ran away.
Lily was sad and went to her friend, Timmy. "Car, Timmy stole my doll. She is not really food. Can you help me find her?" she asked.
Timmy said, "I don't know where the gate is. Itards me to look for it. Do you know where the gate is?"
Lily said, "Yes, I do. We have to go the playground with the swings and the slide."
Timmy said, "That's brave. We can ask the gate if there is a ball that someone dropped."
Lily and Timmy went to the playground and asked the carder if there was a ball

--- SFT | reward 0.59 | stop 1.00  words 0.38 | 114 words
    missed: adven, best, big, disco, goe, hole, live, reali, red, shoe, small, uncom
Once upon a time, there was a little mouse named Mimi. Mimi loved to play and hide all day. One day, Mimi went outside to find some food to eat.
As Mimi walked through the area, she found an wheat. She was excited and didn't know what to do. So, she ran back home to tell her mom.
Her mom came over and said, "What do you have there, Mimi?" Mimi showed her mom the wheat and said, "I found it outside! It's a good place to find food!" So, Mimi and her mom went back outside and went back home. Mimi was happy she found the wheat and could play all day long. 



--- DPO | reward 0.88 | stop 1.00  words 0.82 | 239 words
    missed: adven, disco, goe, play
Once upon a time, there was a little mouse named Mimi. Mimi loved to find food all day long. One day, Mimi went out to find food and find it taste very hot. Mimi felt a little uncomfortable because it was all the way the wheat.
As Mimi was looking for food, she met a big red shoe. The shoe said, "Hello Mimi, why are you so red and hot? Do you need help?" Mimi replied, "Yes, let's find some food together. It will be fun to find what makes it easy to find food."
Mimi and the shoe went to find food, but then they found a big red shoe on the ground. Mimi took the shoe and said, "This shoe is not like the big red shoe. It is the best place for me to live." In the end, Mimi realized the big red shoe was the best place for her to live.
Mimi thought about the big red shoe and decided not to find the big red shoe. She found another small hole, and the plate was full of wheat wheat. Mimi and the n untan

--- SFT | reward 0.52 | stop 1.00  words 0.53  sentence 0.00 | 141 words
    missed: anna, becom, close, overc, picni
Once there were two friends, Sally and Lucy, who loved to play together. They often laughed out loud and had fun. Every day, they would go to the park together - they were always so happy.
But one day, Lucy got very shy of going to the park. Her mom and dad tried to stay friendly and wouldn't make her play. Lucy was so sad!
Her dad had an idea to make her feel better and said, “I will make you smile!”
So her dad cooked dinner together all morning and provided Lucy with lots of delicious food. While they were eating, Lucy felt better, and she liked it stored away.
From then on, Lucy and Lucy were sure that whenever they had company, they wouldn’t be alone. And the shyness was often available for them to explore the park together. 



--- DPO | reward 0.52 | stop 1.00  words 0.54  sentence 0.00 | 248 words
    missed: becom, close, overc, shyne
Once upon a time, there were two people, Sally and Anna. Lucy was a shy girl who didn’t like to smile. Lily was always reminding her polite attitude and always stayed quiet in her heart.
One day, Lucy found a basket of yummy food. She brought it to Lucy and they sat together at a nearby picnic table. Lucy suggested they have a picnic lunch while Lily said, “I have a picnic! We can have lunch and you can slice it with some toast! That way we can both enjoy it!"
Lucy smiled but then admitted, saying, “I shouldn�ted be shy! I don't have much caught up to you. Mummy is giving me a prepare dinner. Can't you finish dropped that?”
Lucy nodded, saying softly, “I think so. We can eat, but only a little bit more. Are you J sprinning? I'm sure you'll like it. Daddy always gives them surprise when you have a chance. We can just enjoy the picnic too!�
Lily smiled again. She said, �That's 

## 3. The same four prompts, greedy

Deterministic, so these don't depend on the seed.

Result: DPO wins all 4 (0.93 / 0.94 / 0.84 / 0.46 vs 0.00 / 0.79 / 0.69 / 0.39). The failure mode is clearest here: **repeating the rare target phrases**. "big red shoe" appears 9 times in the Mimi story, "Look, Dad! A fast car!" 4 times. Rarity weighting pays most for exactly those words.

In [4]:
for i in picks:
    show(pool[i], temperature=0.0)
    print("=" * 100)

Words: point, stadium, fast
Summary: Tom, Lily and their dad go to the stadium to watch cars race. They cheer for their favorite cars and have a fun time.
Features: Dialogue
Story:



--- SFT | reward 0.00 | stop 0.00  words 0.28  dialogue 1.00 | 299 words
    missed: cheer, dad, favor, point, race, stadi, time, watch
Tom and Lily were playing with their toy cars on the floor. They liked to make them go fast and make loud noises. They had a big track that went around the room. They pretended they were racing drivers and made loud noises.
"Look, Lily, I can go faster than you!" Tom said, as he pushed his red car ahead of Lily's blue car.
"No, you can't, Tom! I can go faster than you!" Lily said, as she pushed her blue car harder.
They both laughed and had fun. But then, something bad happened. Tom's blue car hit Lily's blue car and made it crash into Tom's blue car. Tom was very angry and sad. He picked up his blue car and threw it on the floor.
"Hey, what are you doing, Tom? That's not nice!" Lily said, as she saw what Tom did. "You hurt my blue car!"
"I'm sorry, Lily. I didn't mean to hurt your car. I just wanted to play with it." Tom said, as he looked at his blue

--- DPO | reward 0.93 | stop 1.00  words 0.87  dialogue 1.00 | 166 words
    missed: cheer, lily
Once upon a time, there was a little boy named Tom and his dad. They went to the big stadium to watch cars race. Tom was very fast, and he loved to point at the cars. "Look, Dad! A fast car!" he said.
Tom's dad said, "Yes, Tom, that's a fast car. Let's go see the cars race!" They ran to the stadium and saw many cars. Tom pointed at the cars and said, "Look, Dad! A fast car!"
As they watched the cars race, Tom saw a big red car. "Look, Dad! A big red car!" he said. His dad smiled and said, "Yes, Tom, that's a fast car. Let's go see it!"
They ran to the big red car and saw the fast car. Tom pointed at the big red car and said, "Look, Dad! A fast car!" They laughed and clapped for the fast car. They had a fun day at the stadium, and they were happy to see their favorite cars race. 

Features: Dialogue, MoralValue
Words: steal, open, different
Summary: Lily's favorite doll was stolen while she 

--- SFT | reward 0.79 | stop 1.00  words 0.58  dialogue 1.00 | 198 words
    missed: diffe, open, park, steal, stole, taugh
Once upon a time, there was a little girl named Lily. She had a favorite doll that she loved to play with every day. One day, Lily's friend Timmy came to play with her. Timmy saw the doll and wanted to play with it too.
Lily said, "No, Timmy! This is my doll. You can't play with it."
Timmy got upset and said, "But I want to play with it too!"
Lily thought for a moment and said, "Okay, but you have to give it back to me."
Timmy agreed and took the doll from Lily. But when he tried to play with it, he accidentally broke it. Lily was upset and said, "Timmy, you broke my doll! I don't want to play with you anymore."
Timmy felt bad and said, "I'm sorry, Lily. I didn't mean to break your doll. Can we still be friends?"
Lily thought about it and said, "Okay, we can still be friends. But next time, please ask before taking something that doesn't belong to you."
Timmy nodd

--- DPO | reward 0.94 | stop 1.00  words 0.87  dialogue 1.00 | 194 words
    missed: open, taugh
Once upon a time, there was a little girl named Lily. She had a favorite doll named Lily who was very different from her other dolls. One day, Lily's friend Timmy came to the park to play with her. Timmy saw Lily's doll and wanted to steal it.
"Hey Lily, that's my doll!" Timmy said.
Lily replied, "No, it's mine. I found it first."
Timmy got angry and said, "If you don't give me your doll, I will steal it!"
Lily thought about it and said, "Okay, I will give you my doll."
But when Lily went to get her doll, Timmy saw that Lily had stolen it. He said, "I stole your doll. I don't want to steal it."
Lily was very sad and said, "I'm sorry, Timmy. I didn't know it was yours."
Timmy forgave Lily and said, "It's okay, Lily. We all make mistakes. It's important to ask before taking something that doesn't belong to us."
From that day on, Lily learned that it's important to ask before taking something 

--- SFT | reward 0.69 | stop 1.00  words 0.54 | 136 words
    missed: adven, disco, food, goe, hole, place, reali, small, uncom
Once upon a time, there was a little mouse named Mimi. Mimi loved to play in the wheat fields. One day, she found a big red shoe. Mimi thought it was a fun toy to play with.
Mimi took the red shoe to her home. She wanted to show her mom and dad. They were very surprised. They did not know that the red shoe was the best thing for Mimi to play with.
Mimi played with the red shoe all day. She rolled it, jumped on it, and even tried to find it. But then, something unexpected happened. The red shoe started to move! It was not a shoe at all. It was a big, red shoe for Mimi to live in. Mimi was very surprised, but she was happy to have a new friend. 



--- DPO | reward 0.84 | stop 1.00  words 0.76 | 181 words
    missed: best, disco, goe, place, reali
Once upon a time, there was a little mouse named Mimi. Mimi loved to find food in the wheat. One day, she went on an adventure to find some food.
As Mimi walked, she met a big red shoe. The shoe said, "Mimi, I am uncomfortable. I can't find any food." Mimi wanted to help the shoe, so she said, "I will help you find food, but you must promise to be a good mouse."
Mimi and the shoe went on a long walk. They found a big red shoe. Mimi was happy and said, "This is my home!" But then, something unexpected happened. The big red shoe started to move! It was not a shoe, but a big red shoe!
Mimi was scared, but she remembered her small hole. She said, "I am not a big red shoe. I am a big red shoe!" The big red shoe said, "I am not a big red shoe. I am a big red shoe for you to live in." Mimi and the big red shoe became friends and played together every day. 

Summary: Sally overcomes her shyness

--- SFT | reward 0.39 | stop 1.00  words 0.29  sentence 0.00 | 159 words
    missed: anna, becom, close, dinne, frien, overc, shyne, smile, toget
Once upon a time, there was a shy little girl named Sally. She was always so quiet and didn’t like to talk to anyone. One day, Sally was walking to the park with her mom. As they walked, Sally noticed a big picnic basket on the side of the road. She was so excited and ran over to it.
When Sally got to the picnic basket, she saw that it was full of yummy food. She smiled and said, “Let’s have a picnic!” Her mom smiled and said, “Yes, let’s have a picnic!”
Sally and her mom spread out the blanket and started to eat. They laughed and talked and had a wonderful time. After they finished eating, Sally said, “Let’s go home now.” Her mom smiled and said, “Yes, let’s go home.”
Sally smiled and said, “Let’s go home.” Her mom smiled and said, “Yes, let’s go home.”
The two of them walked back home, happy and content. 



--- DPO | reward 0.46 | stop 1.00  words 0.41  sentence 0.00 | 212 words
    missed: becom, close, dinne, overc, shyne, smile
Once upon a time, there was a shy little girl named Sally. Sally was always too scared to talk to anyone. One day, Sally decided to go to the park by herself. As she was walking, Lucy noticed a picnic table with a picnic table. She smiled and said, “Let’s have a picnic! I�ll bring my lunch and then we can have a picnic together. Would you like that?"
Sally was so excited that she smiled and said, “Yes! I would love that!” So, Sally and Anna had a picnic together. They ate sandwiches, fruit, and cookies. After they were done, Sally said, “Let’s go home now. We can have a picnic tomorrow. Is that okay?”
Lucy smiled and said, “Yes, that’s right. I�m so glad you�re here. I'm sure you will. Now, let's go home and have a picnic. We can have a picnic together tomorrow. What do you say?"
Sally smiled and said, "Yes! That sounds like a great idea. I'm so happy that I'm n

## 4. A prompt with a Random sentence and Dialogue

Every check at once: words, sentence, dialogue, stop.

Result: 0.59 vs 0.57, noise. Neither writes the Random sentence, in any sample here; that check has not moved.

In [5]:
full = next(p for p in pool if "Random sentence:" in p and "Dialogue" in p)
show(full, seed=0)

Summary: Anna's favorite doll, Lucy, is taken by her cousin Ben who thinks dolls are for babies. Anna is sad and scared but eventually gets Lucy back and realizes how special and fun she is.
Random sentence: But Anna felt that Lucy understood.
Words: let, luxury, difficult
Features: Dialogue, Foreshadowing
Story:



--- SFT | reward 0.57 | stop 1.00  words 0.42  sentence 0.00  dialogue 1.00 | 227 words
    missed: babi, back, ben, cousi, diffi, event, fun, how, let, reali, sad, scar, taken
Anna loves to play with her dolls. She has many dolls, but her favorite is Lucy. Lucy has long hair, a pretty dress, and a smile. Anna takes Lucy everywhere she go.
One day, Anna's mom takes her to the toy store. She wants to buy a luxury toy. A luxury toy is a big ball on the shelf. It is very expensive and special. Anna's mom says it is a luxury toy. A luxury toy is a small doll that makes you happy.
Anna looks at the luxury toy. It is very pretty. It has colorful face, upsand, and a toy dress. It has sparkles, hugs, and facts. It has a lot of words, but not all. Anna feels that she wants the luxury doll. She wants Lucy for Lucy too.
"Mommy, can I have this doll?" Anna asks. She requests the luxury toy for Lucy.
"Sure, you can have this luxury toy," her mom says. She gives Anna the luxury toy and the spoiled o

--- DPO | reward 0.59 | stop 1.00  words 0.47  sentence 0.00  dialogue 1.00 | 233 words
    missed: babi, back, ben, cousi, diffi, event, luxur, scar, taken
Once upon a time there was a little girl named Anna. Anna had a favorite doll named Lucy who she loved very much. One day, Anna and Lucy were playing together in Dongb cannot find Lucy anywhere. Anna's best friend, Lucy, was very sad because she did not hug Anna very much. Lucy said, "I am sorry about that". Anna felt that Lucy did not understand the importance of both of them.
Suddenly, Anna had an idea. "Let's let Lucy have her new doll. I think it will be fun!" Lucy began to cry and Sam came up to her. "Don't worry," Sam said. "I know how much you like to play with Lucy. She is very special and really amazing."
At first, Anna was confused. She did not understand what anyone meant, but she decided to try her best. Suddenly, Anna felt something easy on Lucy's face. Everyone around her realised that theseweights were not meant for 